# ViT Fine-Tuning: Brain Tumor MRI Classification
Fine-tunes `google/vit-base-patch16-224` on the Brain Tumor MRI dataset (glioma, meningioma, pituitary, notumor).

**Before running:** enable Internet and a T4 GPU in Notebook Settings, and add the dataset `masoudnickparvar/brain-tumor-mri-dataset`.

In [ ]:
!pip install -q transformers torch torchvision scikit-learn matplotlib seaborn

In [ ]:
import os, json
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from transformers import ViTForImageClassification, ViTImageProcessor, get_linear_schedule_with_warmup
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

TRAIN_DIR = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
TEST_DIR = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'

## Model and Preprocessing

In [ ]:
CHECKPOINT = 'google/vit-base-patch16-224'
NUM_CLASSES = 4

image_processor = ViTImageProcessor.from_pretrained(CHECKPOINT)
model = ViTForImageClassification.from_pretrained(
    CHECKPOINT, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
)

## Dataset and DataLoader

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, root, processor):
        self.data = datasets.ImageFolder(root)
        self.processor = processor
        self.classes = self.data.classes

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image, label = self.data[idx]
        inputs = self.processor(images=image.convert('RGB'), return_tensors='pt')
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'labels': torch.tensor(label)
        }

train_dataset = BrainTumorDataset(TRAIN_DIR, image_processor)
test_dataset = BrainTumorDataset(TEST_DIR, image_processor)
CLASS_NAMES = train_dataset.classes

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print('Classes:', CLASS_NAMES)
print(f'Train samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')

## Notebook Requirement 1: Class Distribution Bar Chart

In [ ]:
import numpy as np
train_labels = [label for _, label in train_dataset.data.samples]
counts = np.bincount(train_labels)

plt.figure(figsize=(6,4))
plt.bar(CLASS_NAMES, counts, color='steelblue')
plt.title('Training Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

## Training Loop (6 epochs)

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

EPOCHS = 6
LR = 2e-5

model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
warmup_steps = total_steps // 10
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

history = {'loss': [], 'acc': []}

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            preds = outputs.logits.argmax(dim=-1)
            correct += (preds == batch['labels']).sum().item()
            total += batch['labels'].size(0)

    acc = correct / total
    avg_loss = total_loss / len(train_loader)
    history['loss'].append(avg_loss)
    history['acc'].append(acc)
    print(f'Epoch {epoch+1}/{EPOCHS}  Loss: {avg_loss:.4f}  Test Acc: {acc:.4f}')

## Notebook Requirement 2: Loss and Accuracy Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, EPOCHS+1), history['loss'], marker='o')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(range(1, EPOCHS+1), history['acc'], marker='o', color='green')
axes[1].set_title('Test Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
plt.tight_layout()
plt.show()

## Notebook Requirement 3: Confusion Matrix

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix (Test Set)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## Notebook Requirement 4: Final Test Accuracy

In [ ]:
final_acc = history['acc'][-1]
print(f'FINAL TEST ACCURACY: {final_acc:.4f} ({final_acc*100:.2f}%)')

## Save Artifacts

In [ ]:
os.makedirs('artifacts', exist_ok=True)
torch.save(model.state_dict(), 'artifacts/vit_brain_tumor.pt')

class_names = train_dataset.classes
with open('artifacts/class_names.json', 'w') as f:
    json.dump(class_names, f)

print('Saved:', class_names)
print('Download vit_brain_tumor.pt and class_names.json from the Output panel.')